# Streamlit Demo — Loan Approval Prediction

Sanity-checks the trained model against a handful of representative applicants before it gets wired into `app/app.py`, optionally launches the Streamlit app directly from this notebook for a quick demo, and closes with what a real production deployment would need beyond this demo.

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone -q https://github.com/AnaNicuesa/loan-approval-prediction.git
    %cd loan-approval-prediction
    %pip install -q -r requirements.txt
    %pip install -q pyngrok

In [2]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    path = start.resolve()
    for _ in range(4):
        if (path / "src").exists() and (path / "data").exists():
            return path
        path = path.parent
    raise FileNotFoundError("Could not locate project root from " + str(start))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
import joblib
import pandas as pd

from src.predict import DEFAULT_MODEL_PATH, build_input_frame, predict_loan_status
from src.utils import EXAMPLE_APPLICANTS, ensure_directories

ensure_directories()
model = joblib.load(DEFAULT_MODEL_PATH)
type(model.named_steps["classifier"]).__name__

'RandomForestClassifier'

## Sanity-checking predictions

Three applicants meant to represent the range the model will see in production: a strong candidate, a weak one, and a borderline case where income is reasonable but credit history is missing. These are the same three profiles used for the SHAP local explanations in `04_Model_Evaluation.ipynb`, defined once in `src/utils.py` so both notebooks stay in sync.

In [4]:
for name, applicant in EXAMPLE_APPLICANTS.items():
    frame = build_input_frame(**applicant)
    result = predict_loan_status(model, frame)
    print(f"{name}: {result['label']} (probability approved = {result['probability_approved']:.3f})")

strong_applicant: Approved (probability approved = 0.948)
weak_applicant: Rejected (probability approved = 0.135)
borderline_applicant: Approved (probability approved = 0.926)


The strong and weak candidates come out exactly as expected given the credit history effect seen throughout the earlier notebooks. The borderline candidate turns out less ambiguous than its name suggests -- see the SHAP breakdown in `04_Model_Evaluation.ipynb`, Section 10, for the full explanation of which factors actually drive that prediction.

## Running the app

Locally, from the project root:

```bash
pip install -r requirements.txt
streamlit run app/app.py
```

The cell below launches the same app from within Colab and exposes it through an ngrok tunnel — useful for a live demo without a local install. It's skipped automatically outside of Colab.

In [5]:
import subprocess
import time

if IN_COLAB:
    from pyngrok import ngrok

    streamlit_process = subprocess.Popen(
        ["streamlit", "run", "app/app.py", "--server.port", "8501", "--server.headless", "true"]
    )
    time.sleep(5)
    public_url = ngrok.connect(8501)
    print(f"Streamlit app running at: {public_url}")
else:
    print("Not running in Colab — start the app locally with `streamlit run app/app.py`.")

Not running in Colab — start the app locally with `streamlit run app/app.py`.


## How this would look in production

Streamlit is a demo tool here, not a production architecture. It's a single Python process that reloads the model into memory, has no authentication, no rate limiting, no structured logging, and no separation between the UI and the prediction logic -- fine for a portfolio demo and a live walkthrough, wrong for anything serving real traffic or real lending decisions.

A production version of this system would look closer to:

```
User
  |
  v
API layer (FastAPI or Flask)
  |
  v
Preprocessing pipeline (same ColumnTransformer, versioned)
  |
  v
Model (same best_model.joblib, versioned)
  |
  v
Prediction (+ SHAP explanation, computed the same way as here)
  |
  v
Logging (every request, prediction, and explanation, for audit)
  |
  v
Monitoring (feature drift, prediction drift, and latency, alerting on all three)
```

Concretely, what changes and what doesn't:

- **Doesn't change**: `src/preprocessing.py`, `src/predict.py`, and `src/explain.py` are already decoupled from the app -- they'd move into the API layer unmodified. That decoupling isn't an accident; it's why the same prediction and explanation code runs in this notebook, in `app/app.py`, and could run behind a FastAPI endpoint with no rewrite.
- **Changes**: the model and preprocessing pipeline need version pinning (which `models/best_model.joblib` was this prediction served from?), every request/response needs to be logged for audit and dispute resolution, and there needs to be active monitoring for both data drift (are incoming applications starting to look different from training data?) and prediction drift (is the approval rate shifting over time in a way that wasn't intended?).
- **New requirement**: a human-in-the-loop path for the borderline and rejected cases discussed in `04_Model_Evaluation.ipynb`'s Business Perspective section -- a production system that fully automates lending decisions from a 510-row model would be taking on risk the model was never validated to carry.

None of this is implemented in this repository -- it's out of scope for a portfolio project -- but knowing the gap between "a working Streamlit demo" and "a production lending system" is itself part of the deliverable.

## Summary

- `models/best_model.joblib` produces sensible, well-calibrated predictions across strong, weak, and borderline applicant profiles.
- `src/predict.py` is the single source of truth for turning form inputs into a prediction, shared by this notebook and `app/app.py` — no logic is duplicated between them.
- The app is ready to run either locally or directly from Colab via the ngrok tunnel above.
- Streamlit is explicitly a demo layer -- the API-based architecture above is what a real deployment would require instead.

> **Potential Interview Questions**
>
> **Q: Why Streamlit instead of a real API?**
> A: Streamlit is fast to build and great for a live demo or portfolio review, but it bundles UI and business logic into one process with no auth, no logging, and no horizontal scalability. It was the right tool for this project's actual goal -- an interactive demo -- not a claim that it's production-ready.
>
> **Q: What would break first if this had to handle real traffic?**
> A: The model is reloaded from disk into a single process with no request queue, no load balancing, and no caching strategy beyond Streamlit's own `st.cache_resource`. There's also no logging of predictions, so there'd be no way to audit a disputed decision or detect drift after the fact.
>
> **Q: How would you monitor this model after deployment?**
> A: Two separate things: data drift (are incoming applicant profiles starting to look statistically different from the training distribution?) and prediction/outcome drift (is the approval rate or the SHAP attribution pattern shifting over time?). Both need a logging pipeline in place from day one, since you can't retroactively monitor drift you didn't record.
>
> **Q: Why keep `src/predict.py` and `src/explain.py` separate from `app/app.py` instead of writing the app as one self-contained script?**
> A: So the exact same prediction and explanation logic can move behind a FastAPI endpoint later without being rewritten or drifting out of sync with what's been validated in the notebooks. It also means this notebook, the app, and any future API all call one tested implementation instead of three copies of similar-but-not-identical logic.
>
> **Q: If you had one week to make this production-ready, what would you prioritize first?**
> A: Logging every prediction with its inputs, model version, and SHAP explanation -- before anything else. Without that, there's no way to audit a decision after the fact or detect drift later, and every other production concern (monitoring, retraining triggers, A/B testing a new model) depends on that data existing.